# Proyecto final · Minería de Datos 1
## Dataset 1: Calificaciones de estudiantes (Student Grade SES)
### 02. EDA – Análisis estadístico

**Fuente de trabajo:** `data/processed/calificaciones_limpio.csv`  
**Alcance:** análisis estadístico descriptivo; las visualizaciones se desarrollarán en `03_EDA_Visual.ipynb`.

**Introducción**

El análisis exploratorio estadístico (EDA) busca comprender la distribución del rendimiento académico y su relación descriptiva con características socioeconómicas, familiares, escolares y hábitos de estudio. Se utilizará la versión limpia del dataset obtenida en la etapa ETL, sin modificarla.

**Objetivos**

- Describir el comportamiento de las variables mediante medidas estadísticas, frecuencias y porcentajes.
- Examinar el rendimiento académico por grupos y posibles factores asociados.
- Identificar patrones relevantes y limitaciones para la siguiente etapa de visualización y preprocesamiento.
- Elaborar un resumen verificable y conclusiones sustentadas en los resultados del análisis.

## Parte 02. EDA – Análisis estadístico

### 1. Importación de librerías

In [1]:
from pathlib import Path
import unicodedata
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", "{:,.2f}".format)
print("Pandas:", pd.__version__, "| NumPy:", np.__version__)

Pandas: 3.1.1 | NumPy: 2.5.4


### 2. Carga del dataset limpio
Se utiliza únicamente el CSV procesado. Se conservan sus datos originales y se crea una copia de trabajo con nombres internos estandarizados para facilitar el análisis.

In [2]:
archivo = "calificaciones_limpio.csv"
actual = Path.cwd().resolve()
raiz = next((p for p in [actual, *actual.parents] if (p / "data" / "processed").is_dir()), None)
ruta_windows = Path(r"C:\Users\andre\Documents\CUC\Cuatrimeste_lll_2026\MineriaDatos\Proyecto\Mineria_Datos_1\data\processed\calificaciones_limpio.csv")
rutas = ([raiz / "data" / "processed" / archivo] if raiz else []) + [ruta_windows]
ruta_csv = next((p for p in rutas if p.is_file()), None)
if ruta_csv is None:
    raise FileNotFoundError(f"No se encontró {archivo}. Se revisaron: {[str(p) for p in rutas]}")

# Leer sin sobrescribir el CSV.
df = pd.read_csv(ruta_csv, encoding="utf-8-sig", low_memory=False)

# Orden original de las 29 variables del dataset Student Grade SES.
columnas_originales = [
    "student_id", "academic_year", "survey_date", "age", "gender",
    "grade_level", "school_type", "residence_type", "family_size", "num_siblings",
    "single_parent_household", "family_income_bracket", "annual_family_income_inr",
    "father_education_level", "mother_education_level", "father_occupation", "mother_occupation",
    "has_internet_access", "has_computer_or_laptop", "distance_to_school_km",
    "scholarship_status", "extra_tutoring", "parental_involvement_score",
    "study_hours_per_week", "attendance_rate_pct", "previous_year_grade_pct",
    "final_grade_pct", "grade_class", "at_risk_flag"
]

# Si el ETL cambió los nombres al español, identificarlos por la posición
# ORIGINAL de las columnas (el ETL renombró sin alterar ese orden).
if set(columnas_originales).issubset(df.columns):
    mapa_columnas = {c: c for c in columnas_originales}
elif df.shape[1] == len(columnas_originales):
    mapa_columnas = dict(zip(df.columns.tolist(), columnas_originales))
    print("AVISO: equivalencias por posición. Comprobá la tabla para confirmar el orden del ETL.")
else:
    raise ValueError(
        "Se esperaban 29 columnas. Revisá si se está leyendo el archivo limpio correcto."
    )

base = df.rename(columns=mapa_columnas).copy()
print("Archivo cargado:", ruta_csv)
print("Dimensiones:", base.shape)
display(pd.DataFrame({"Columna del CSV": list(mapa_columnas.keys()),
                      "Nombre interno para el análisis": list(mapa_columnas.values())}))

AVISO: equivalencias por posición. Comprobá la tabla para confirmar el orden del ETL.
Archivo cargado: C:\Users\andre\Documents\CUC\Cuatrimeste_lll_2026\MineriaDatos\Proyecto\Mineria_Datos_1\data\processed\calificaciones_limpio.csv
Dimensiones: (12000, 29)


,Columna del CSV,Nombre interno para el análisis
0,id_estudiante,student_id
1,periodo_academico,academic_year
2,fecha_encuesta,survey_date
3,edad,age
4,genero,gender
5,nivel_academico,grade_level
6,tipo_escuela,school_type
7,zona_residencia,residence_type
8,tamano_familia,family_size
9,numero_hermanos,num_siblings


### 3. Verificación de dimensiones y estructura
Se revisa que el archivo cargado tenga la estructura esperada antes del análisis, sin realizar ninguna limpieza adicional.

In [3]:
verificacion = pd.DataFrame({
    "Indicador": ["Registros", "Columnas", "Valores nulos", "Filas completamente duplicadas",
                  "Identificadores repetidos"],
    "Valor": [len(base), len(base.columns), int(base.isna().sum().sum()),
              int(base.duplicated().sum()), int(base["student_id"].duplicated().sum())]
})
display(verificacion)
print("Tipos de datos:")
display(base.dtypes.rename("Tipo").to_frame())

if len(base) != 12000 or base.shape[1] != 29:
    print("ADVERTENCIA: dimensiones diferentes a las esperadas (12.000 × 29).")
if base.isna().any().any():
    print("ADVERTENCIA: se encontraron nulos. Revisá el CSV exportado desde el ETL.")

,Indicador,Valor
0,Registros,12000
1,Columnas,29
2,Valores nulos,0
3,Filas completamente duplicadas,0
4,Identificadores repetidos,0


Tipos de datos:


,Tipo
student_id,str
academic_year,str
survey_date,str
age,int64
gender,str
grade_level,int64
school_type,str
residence_type,str
family_size,int64
num_siblings,int64


### 4. Clasificación de variables para el análisis estadístico
La clasificación considera el significado de las variables. Un indicador representado por 0 y 1 se trata como categoría, no como una medida cuantitativa.

In [4]:
etiquetas = {
    "student_id":"ID del estudiante", "academic_year":"Año académico", "survey_date":"Fecha de encuesta",
    "age":"Edad", "gender":"Género", "grade_level":"Grado académico", "school_type":"Tipo de centro educativo",
    "residence_type":"Zona de residencia", "family_size":"Tamaño familiar", "num_siblings":"Número de hermanos",
    "single_parent_household":"Hogar monoparental", "family_income_bracket":"Categoría de ingreso familiar",
    "annual_family_income_inr":"Ingreso familiar anual (INR)", "father_education_level":"Educación del padre",
    "mother_education_level":"Educación de la madre", "father_occupation":"Ocupación del padre",
    "mother_occupation":"Ocupación de la madre", "has_internet_access":"Acceso a internet",
    "has_computer_or_laptop":"Tiene computadora", "distance_to_school_km":"Distancia escolar (km)",
    "scholarship_status":"Estado de beca", "extra_tutoring":"Tutorías adicionales",
    "parental_involvement_score":"Participación parental", "study_hours_per_week":"Horas de estudio semanales",
    "attendance_rate_pct":"Asistencia (%)", "previous_year_grade_pct":"Calificación anterior (%)",
    "final_grade_pct":"Calificación final (%)", "grade_class":"Categoría de calificación",
    "at_risk_flag":"Indicador de riesgo académico"
}

numericas = ["age", "family_size", "num_siblings", "annual_family_income_inr",
            "distance_to_school_km", "parental_involvement_score", "study_hours_per_week",
            "attendance_rate_pct", "previous_year_grade_pct", "final_grade_pct"]
categoricas = [
    "academic_year", "gender", "grade_level", "school_type", "residence_type",
    "single_parent_household", "family_income_bracket", "father_education_level",
    "mother_education_level", "father_occupation", "mother_occupation",
    "has_internet_access", "has_computer_or_laptop", "scholarship_status",
    "extra_tutoring", "grade_class", "at_risk_flag"
]
clasificacion = pd.DataFrame({
    "Variable": base.columns,
    "Descripción": [etiquetas.get(x,x) for x in base.columns],
    "Naturaleza": ["Identificador" if x == "student_id" else
                    "Fecha" if x == "survey_date" else
                    "Cuantitativa" if x in numericas else
                    "Categórica / ordinal" if x in categoricas else "Revisar"
                    for x in base.columns],
    "Tipo Pandas": base.dtypes.astype(str).to_numpy()
})
display(clasificacion)
print("Variables numéricas:", len(numericas), "| Categóricas / ordinales:", len(categoricas))

# Algunas columnas pueden contener 'Desconocido'; se conservan como categorías.
for col in numericas:
    base[col] = pd.to_numeric(base[col], errors="coerce")

objetivo = "final_grade_pct"
# Estas columnas pueden depender de la nota final; NO son factores independientes.
variables_derivadas_objetivo = ["grade_class", "at_risk_flag"]

,Variable,Descripción,Naturaleza,Tipo Pandas
0,student_id,ID del estudiante,Identificador,str
1,academic_year,Año académico,Categórica / ordinal,str
2,survey_date,Fecha de encuesta,Fecha,str
3,age,Edad,Cuantitativa,int64
4,gender,Género,Categórica / ordinal,str
5,grade_level,Grado académico,Categórica / ordinal,int64
6,school_type,Tipo de centro educativo,Categórica / ordinal,str
7,residence_type,Zona de residencia,Categórica / ordinal,str
8,family_size,Tamaño familiar,Cuantitativa,int64
9,num_siblings,Número de hermanos,Cuantitativa,int64


Variables numéricas: 10 | Categóricas / ordinales: 17


### 5. Estadísticas descriptivas de variables numéricas
Se presenta una vista general de las medidas estadísticas. Se profundizará en tendencia central, dispersión y cuartiles en los apartados siguientes.

In [5]:
estadisticas_generales = base[numericas].describe().T
estadisticas_generales.index = [etiquetas[x] for x in estadisticas_generales.index]
display(estadisticas_generales)

,count,mean,std,min,25%,50%,75%,max
Edad,"12,000.00",13.99,2.15,10.00,12.00,14.00,16.00,18.00
Tamaño familiar,"12,000.00",4.60,1.40,2.00,4.00,5.00,6.00,10.00
Número de hermanos,"12,000.00",2.13,1.42,0.00,1.00,2.00,3.00,8.00
Ingreso familiar anual (INR),"12,000.00","643,253.90","919,793.61","60,010.00","193,876.50","351,958.00","653,176.50","19,621,849.00"
Distancia escolar (km),"12,000.00",3.87,3.55,0.20,1.50,2.80,5.10,31.20
Participación parental,"12,000.00",3.25,0.94,1.00,3.00,3.00,4.00,5.00
Horas de estudio semanales,"12,000.00",14.43,5.02,0.00,11.30,14.30,17.30,65.00
Asistencia (%),"12,000.00",85.86,7.86,57.80,80.40,86.00,91.50,100.00
Calificación anterior (%),"12,000.00",62.34,10.52,23.50,55.38,62.20,69.30,100.00
Calificación final (%),"12,000.00",66.94,11.70,2.80,60.10,67.30,74.60,100.00


### 6. Análisis de medidas de tendencia central (media, mediana y moda)
La moda de una variable continua puede ser poco representativa si numerosos valores aparecen una sola vez.

In [6]:
tendencia = []
for col in numericas:
    s = base[col].dropna()
    modas = s.mode()
    tendencia.append({"Variable": etiquetas[col], "Media": s.mean(), "Mediana": s.median(),
                      "Moda (primera)": modas.iloc[0] if len(modas) else np.nan,
                      "Repeticiones de moda": int(s.value_counts().max()) if len(s) else 0})
tabla_tendencia = pd.DataFrame(tendencia)
display(tabla_tendencia)

,Variable,Media,Mediana,Moda (primera),Repeticiones de moda
0,Edad,13.99,14.00,13.00,1749
1,Tamaño familiar,4.60,5.00,5.00,3239
2,Número de hermanos,2.13,2.00,2.00,3135
3,Ingreso familiar anual (INR),"643,253.90","351,958.00","252,931.00",204
4,Distancia escolar (km),3.87,2.80,1.70,358
5,Participación parental,3.25,3.00,3.00,4767
6,Horas de estudio semanales,14.43,14.30,15.30,124
7,Asistencia (%),85.86,86.00,100.00,515
8,Calificación anterior (%),62.34,62.20,63.30,155
9,Calificación final (%),66.94,67.30,60.90,58


### 7. Análisis de medidas de dispersión (varianza, desviación estándar y rango)
El coeficiente de variación solo se interpreta si la media es distinta de cero y la escala lo permite. Las varianzas utilizan las unidades al cuadrado.

In [7]:
dispersion = []
for col in numericas:
    s = base[col].dropna()
    media = s.mean()
    dispersion.append({
        "Variable": etiquetas[col], "Varianza muestral": s.var(),
        "Desviación estándar": s.std(), "Mínimo": s.min(), "Máximo": s.max(),
        "Rango": s.max()-s.min(),
        "CV (%)": s.std()/abs(media)*100 if pd.notna(media) and media != 0 else np.nan
    })
tabla_dispersion = pd.DataFrame(dispersion)
display(tabla_dispersion)

,Variable,Varianza muestral,Desviación estándar,Mínimo,Máximo,Rango,CV (%)
0,Edad,4.63,2.15,10.00,18.00,8.00,15.39
1,Tamaño familiar,1.95,1.40,2.00,10.00,8.00,30.38
2,Número de hermanos,2.02,1.42,0.00,8.00,8.00,66.66
3,Ingreso familiar anual (INR),"846,020,291,452.31","919,793.61","60,010.00","19,621,849.00","19,561,839.00",142.99
4,Distancia escolar (km),12.59,3.55,0.20,31.20,31.00,91.62
5,Participación parental,0.89,0.94,1.00,5.00,4.00,28.99
6,Horas de estudio semanales,25.21,5.02,0.00,65.00,65.00,34.80
7,Asistencia (%),61.81,7.86,57.80,100.00,42.20,9.16
8,Calificación anterior (%),110.69,10.52,23.50,100.00,76.50,16.88
9,Calificación final (%),136.82,11.70,2.80,100.00,97.20,17.47


### 8. Análisis de cuartiles y rango intercuartílico
Los cuartiles permiten describir la dispersión del 50 % central de los registros, sin eliminar valores extremos.

In [8]:
cuartiles = []
for col in numericas:
    s = base[col].dropna()
    q1, q2, q3 = s.quantile([.25,.50,.75])
    cuartiles.append({"Variable": etiquetas[col], "Q1 (25 % )": q1,
                      "Q2 / Mediana": q2, "Q3 (75 % )": q3,
                      "Rango intercuartílico": q3-q1})
tabla_cuartiles = pd.DataFrame(cuartiles)
display(tabla_cuartiles)

,Variable,Q1 (25 % ),Q2 / Mediana,Q3 (75 % ),Rango intercuartílico
0,Edad,12.00,14.00,16.00,4.00
1,Tamaño familiar,4.00,5.00,6.00,2.00
2,Número de hermanos,1.00,2.00,3.00,2.00
3,Ingreso familiar anual (INR),"193,876.50","351,958.00","653,176.50","459,300.00"
4,Distancia escolar (km),1.50,2.80,5.10,3.60
5,Participación parental,3.00,3.00,4.00,1.00
6,Horas de estudio semanales,11.30,14.30,17.30,6.00
7,Asistencia (%),80.40,86.00,91.50,11.10
8,Calificación anterior (%),55.38,62.20,69.30,13.92
9,Calificación final (%),60.10,67.30,74.60,14.50


### 9. Análisis de asimetría y curtosis
La asimetría muestra hacia qué lado se extiende la distribución; la curtosis de Pandas es el exceso de curtosis (una distribución normal tiene valor aproximado 0).

In [9]:
forma = []
for col in numericas:
    s = base[col].dropna()
    sesgo = s.skew()
    forma.append({
        "Variable": etiquetas[col], "Asimetría": sesgo, "Curtosis (exceso)": s.kurt(),
        "Sentido": "Positiva" if sesgo > 0.5 else "Negativa" if sesgo < -0.5 else "Cercana a simétrica"
    })
tabla_forma = pd.DataFrame(forma)
display(tabla_forma)

,Variable,Asimetría,Curtosis (exceso),Sentido
0,Edad,0.02,-0.94,Cercana a simétrica
1,Tamaño familiar,0.17,-0.22,Cercana a simétrica
2,Número de hermanos,0.33,-0.33,Cercana a simétrica
3,Ingreso familiar anual (INR),4.45,37.87,Positiva
4,Distancia escolar (km),2.06,5.74,Positiva
5,Participación parental,-0.09,-0.36,Cercana a simétrica
6,Horas de estudio semanales,1.95,16.29,Positiva
7,Asistencia (%),-0.19,-0.40,Cercana a simétrica
8,Calificación anterior (%),0.04,0.06,Cercana a simétrica
9,Calificación final (%),-0.53,1.63,Negativa


### 10. Análisis de frecuencias de variables categóricas
Se muestran las frecuencias de cada categoría. Se excluyen el identificador y la fecha de encuesta porque no representan grupos de comparación.

In [10]:
tablas_frecuencia = {}
for col in categoricas:
    serie = base[col].astype("string").fillna("Desconocido")
    frec = serie.value_counts(dropna=False).rename_axis("Categoría").reset_index(name="Frecuencia")
    frec["Porcentaje (%)"] = 100 * frec["Frecuencia"] / len(base)
    tablas_frecuencia[col] = frec
    print(f"\n{etiquetas[col]} | {len(frec)} categorías")
    display(frec.head(15))
    if len(frec)>15:
        print(f"Se muestran las 15 categorías más frecuentes de {len(frec)}.")


Año académico | 4 categorías


,Categoría,Frecuencia,Porcentaje (%)
0,2024-2025,3834,31.95
1,2023-2024,3338,27.82
2,2022-2023,2704,22.53
3,2021-2022,2124,17.70



Género | 3 categorías


,Categoría,Frecuencia,Porcentaje (%)
0,Male,5919,49.33
1,Female,5733,47.77
2,Other,348,2.90



Grado académico | 7 categorías


,Categoría,Frecuencia,Porcentaje (%)
0,9,1798,14.98
1,7,1739,14.49
2,11,1734,14.45
3,8,1712,14.27
4,6,1698,14.15
5,10,1667,13.89
6,12,1652,13.77


... (24 outputs left out)


Categoría de calificación | 5 categorías


,Categoría,Frecuencia,Porcentaje (%)
0,C,6147,51.23
1,D,2740,22.83
2,B,2686,22.38
3,F,218,1.82
4,A,209,1.74



Indicador de riesgo académico | 2 categorías


,Categoría,Frecuencia,Porcentaje (%)
0,0,11782,98.18
1,1,218,1.82


### 11. Análisis de porcentajes y proporciones
Se resumen las principales variables categóricas para comparar la composición del dataset. Una frecuencia elevada no implica que la categoría explique el rendimiento.

In [11]:
categorias_clave = ["gender", "residence_type", "family_income_bracket", "school_type",
                    "scholarship_status", "has_internet_access", "extra_tutoring",
                    "grade_class", "at_risk_flag"]
porcentajes = []
for col in categorias_clave:
    frec = tablas_frecuencia[col]
    for _, r in frec.iterrows():
        porcentajes.append({"Variable": etiquetas[col], "Categoría": r["Categoría"],
                            "Cantidad": int(r["Frecuencia"]),
                            "Porcentaje (%)": r["Porcentaje (%)"]})
tabla_porcentajes = pd.DataFrame(porcentajes)
display(tabla_porcentajes)

,Variable,Categoría,Cantidad,Porcentaje (%)
0,Género,Male,5919,49.33
1,Género,Female,5733,47.77
2,Género,Other,348,2.90
3,Zona de residencia,Urban,5026,41.88
4,Zona de residencia,Rural,4224,35.20
5,Zona de residencia,Semi-Urban,2750,22.92
6,Categoría de ingreso familiar,Lower-Middle,3363,28.02
7,Categoría de ingreso familiar,Middle,3339,27.82
8,Categoría de ingreso familiar,Low,2628,21.90
9,Categoría de ingreso familiar,Upper-Middle,1797,14.97


### 12. Análisis estadístico del rendimiento académico
La calificación final es la variable principal de referencia. `grade_class` y `at_risk_flag` se describen, pero no se interpretan como causas independientes de la calificación.

In [12]:
cal_final = base[objetivo]
resumen_rendimiento = pd.DataFrame({
    "Indicador": ["Estudiantes", "Media (%)", "Mediana (%)", "Desviación estándar",
                  "Mínimo (%)", "Q1 (%)", "Q3 (%)", "Máximo (%)"],
    "Valor": [cal_final.count(), cal_final.mean(), cal_final.median(),
              cal_final.std(), cal_final.min(), cal_final.quantile(.25),
              cal_final.quantile(.75), cal_final.max()]
})
display(resumen_rendimiento)

print("Distribución de la categoría de calificación:")
display(tablas_frecuencia["grade_class"])
print("Distribución del indicador de riesgo:")
display(tablas_frecuencia["at_risk_flag"])

print("Calificación final por categoría (DESCRIPTIVO, no causal):")
display(base.groupby("grade_class", dropna=False)[objetivo]
        .agg(Estudiantes="count", Media="mean", Mediana="median").sort_values("Media", ascending=False))

,Indicador,Valor
0,Estudiantes,"12,000.00"
1,Media (%),66.94
2,Mediana (%),67.30
3,Desviación estándar,11.70
4,Mínimo (%),2.80
5,Q1 (%),60.10
6,Q3 (%),74.60
7,Máximo (%),100.00


Distribución de la categoría de calificación:


,Categoría,Frecuencia,Porcentaje (%)
0,C,6147,51.23
1,D,2740,22.83
2,B,2686,22.38
3,F,218,1.82
4,A,209,1.74


Distribución del indicador de riesgo:


,Categoría,Frecuencia,Porcentaje (%)
0,0,11782,98.18
1,1,218,1.82


Calificación final por categoría (DESCRIPTIVO, no causal):


,Estudiantes,Media,Mediana
grade_class,,,
A,209,93.85,92.90
B,2686,80.31,79.60
C,6147,67.36,67.40
D,2740,53.89,54.90
F,218,28.84,30.75


### 13. Comparación del rendimiento según características socioeconómicas
Las diferencias entre grupos son asociaciones descriptivas. Se muestra el número de estudiantes para valorar su representatividad.

In [13]:
def comparar_grupos(col, minimo=30, max_filas=25):
    tabla = (base.groupby(col, dropna=False)[objetivo]
             .agg(Estudiantes="count", Media="mean", Mediana="median", Desviacion="std")
             .reset_index().rename(columns={col:"Categoría"}))
    tabla["Grupo suficiente"] = tabla["Estudiantes"] >= minimo
    return tabla.sort_values("Media", ascending=False).head(max_filas)

comparaciones_grupos = {}
variables_socio = ["family_income_bracket", "residence_type", "has_internet_access",
                   "has_computer_or_laptop"]
for col in variables_socio:
    comparaciones_grupos[col] = comparar_grupos(col)
    print("\n", etiquetas[col])
    display(comparaciones_grupos[col])


 Categoría de ingreso familiar


,Categoría,Estudiantes,Media,Mediana,Desviacion,Grupo suficiente
0,High,873,76.42,77.10,11.03,True
4,Upper-Middle,1797,71.60,72.20,11.88,True
3,Middle,3339,67.75,68.10,11.13,True
2,Lower-Middle,3363,65.01,65.30,10.91,True
1,Low,2628,62.07,62.20,10.22,True



 Zona de residencia


,Categoría,Estudiantes,Media,Mediana,Desviacion,Grupo suficiente
2,Urban,5026,67.54,68.10,11.83,True
1,Semi-Urban,2750,66.74,67.40,11.74,True
0,Rural,4224,66.37,66.40,11.48,True



 Acceso a internet


,Categoría,Estudiantes,Media,Mediana,Desviacion,Grupo suficiente
1,1.0,5869,68.86,69.40,11.86,True
2,Desconocido,180,67.17,67.95,11.82,True
0,0.0,5951,65.04,65.40,11.21,True



 Tiene computadora


,Categoría,Estudiantes,Media,Mediana,Desviacion,Grupo suficiente
1,1,4991,69.10,69.50,11.82,True
0,0,7009,65.41,65.90,11.36,True


### 14. Comparación del rendimiento según características familiares
Se evalúan diferencias de calificación según contexto familiar. La categoría `Desconocido` señala información faltante imputada durante el ETL y no debe tratarse como un nivel educativo u ocupación real.

In [14]:
variables_familiares = ["single_parent_household", "father_education_level",
                        "mother_education_level", "father_occupation", "mother_occupation",
                        "family_size", "num_siblings", "parental_involvement_score"]
for col in variables_familiares:
    print("\n", etiquetas[col])
    tabla = comparar_grupos(col, minimo=30)
    comparaciones_grupos[col] = tabla
    display(tabla)


 Hogar monoparental


,Categoría,Estudiantes,Media,Mediana,Desviacion,Grupo suficiente
0,0,10318,67.06,67.40,11.75,True
1,1,1682,66.24,66.70,11.37,True



 Educación del padre


,Categoría,Estudiantes,Media,Mediana,Desviacion,Grupo suficiente
2,Doctorate,361,76.03,77.00,12.85,True
4,Master's Degree,1079,73.05,73.30,11.76,True
0,Bachelor's Degree,2387,70.19,70.60,11.20,True
1,Desconocido,420,66.97,67.60,12.15,True
3,Higher Secondary,2099,66.90,67.50,10.99,True
7,Secondary,2827,65.22,65.60,10.88,True
6,Primary,2016,63.20,63.30,10.61,True
5,No Formal Education,811,60.62,60.60,10.71,True



 Educación de la madre


,Categoría,Estudiantes,Media,Mediana,Desviacion,Grupo suficiente
2,Doctorate,379,75.51,77.00,12.09,True
4,Master's Degree,1081,72.96,73.50,11.96,True
0,Bachelor's Degree,2423,70.05,70.30,11.47,True
3,Higher Secondary,2179,67.23,67.80,11.03,True
1,Desconocido,360,66.67,67.20,11.79,True
7,Secondary,2793,65.27,65.60,10.81,True
6,Primary,1963,63.00,63.20,10.49,True
5,No Formal Education,822,60.40,60.60,10.61,True


... (6 outputs left out)


 Número de hermanos


,Categoría,Estudiantes,Media,Mediana,Desviacion,Grupo suficiente
0,0,1720,67.86,68.20,11.98,True
1,1,2478,67.30,67.90,11.70,True
2,2,3135,66.87,67.00,11.79,True
3,3,2634,66.78,67.30,11.41,True
4,4,1404,66.37,66.70,11.50,True
5,5,497,65.63,65.90,11.87,True
6,6,112,64.57,64.85,10.97,True
7,7,17,63.12,65.00,15.28,False
8,8,3,62.50,62.30,14.40,False



 Participación parental


,Categoría,Estudiantes,Media,Mediana,Desviacion,Grupo suficiente
4,5,1035,68.79,69.50,11.69,True
3,4,3712,67.92,68.30,11.90,True
2,3,4767,66.69,67.00,11.49,True
0,1,339,65.28,64.90,11.70,True
1,2,2147,65.20,65.60,11.52,True


### 15. Comparación del rendimiento según hábitos de estudio y asistencia
Para las variables numéricas se forman cuartiles de estudiantes. Estos cuartiles son grupos estadísticos para comparar y no una modificación del CSV.

In [15]:
variables_habitos = ["study_hours_per_week", "attendance_rate_pct", "previous_year_grade_pct"]
comparaciones_cuartiles = {}
for col in variables_habitos:
    columna_grupo = f"grupo_{col}"
    temporal = base[[col, objetivo]].dropna().copy()
    # Etiquetas ordenadas por el nivel de la variable para evitar suponer cortes iguales.
    temporal[columna_grupo] = pd.qcut(temporal[col], q=4, duplicates="drop")
    comparacion = (temporal.groupby(columna_grupo, observed=True)[objetivo]
                   .agg(Estudiantes="count", Media="mean", Mediana="median")
                   .reset_index().rename(columns={columna_grupo:"Intervalo"}))
    comparacion["Intervalo"] = comparacion["Intervalo"].astype(str)
    comparaciones_cuartiles[col] = comparacion
    print("\n", etiquetas[col])
    display(comparacion)

print("\nTutorías adicionales")
comparaciones_grupos["extra_tutoring"] = comparar_grupos("extra_tutoring")
display(comparaciones_grupos["extra_tutoring"])


 Horas de estudio semanales


,Intervalo,Estudiantes,Media,Mediana
0,"(-0.001, 11.3]",3009,63.76,63.80
1,"(11.3, 14.3]",3055,66.03,66.40
2,"(14.3, 17.3]",2989,67.63,68.00
3,"(17.3, 65.0]",2947,70.44,71.10



 Asistencia (%)


,Intervalo,Estudiantes,Media,Mediana
0,"(57.799, 80.4]",3034,64.50,64.90
1,"(80.4, 86.0]",2996,66.12,66.40
2,"(86.0, 91.5]",2980,67.69,68.00
3,"(91.5, 100.0]",2990,69.51,70.10



 Calificación anterior (%)


,Intervalo,Estudiantes,Media,Mediana
0,"(23.499, 55.375]",3000,59.72,60.10
1,"(55.375, 62.2]",3028,64.96,65.35
2,"(62.2, 69.3]",3013,68.49,68.90
3,"(69.3, 100.0]",2959,74.73,75.30



Tutorías adicionales


,Categoría,Estudiantes,Media,Mediana,Desviacion,Grupo suficiente
1,1,2762,69.48,70.00,11.98,True
0,0,9238,66.19,66.50,11.50,True


### 16. Comparación del rendimiento según características escolares
Se revisa el rendimiento según centro educativo, grado, año académico y beca. No se atribuyen diferencias exclusivamente a estas características.

In [16]:
variables_escolares = ["school_type", "grade_level", "academic_year", "scholarship_status"]
for col in variables_escolares:
    comparaciones_grupos[col] = comparar_grupos(col)
    print("\n", etiquetas[col])
    display(comparaciones_grupos[col])


 Tipo de centro educativo


,Categoría,Estudiantes,Media,Mediana,Desviacion,Grupo suficiente
1,Government-Aided,2386,67.00,67.60,11.89,True
0,Government,5373,66.93,67.20,11.74,True
2,Private-Unaided,4241,66.93,67.40,11.54,True



 Grado académico


,Categoría,Estudiantes,Media,Mediana,Desviacion,Grupo suficiente
5,11,1734,67.20,67.20,11.33,True
6,12,1652,67.12,67.60,11.99,True
2,8,1712,67.11,67.60,11.60,True
0,6,1698,67.01,67.45,11.55,True
1,7,1739,66.88,67.00,11.43,True
4,10,1667,66.70,67.30,11.76,True
3,9,1798,66.61,67.00,12.20,True



 Año académico


,Categoría,Estudiantes,Media,Mediana,Desviacion,Grupo suficiente
1,2022-2023,2704,67.14,67.55,11.78,True
0,2021-2022,2124,66.98,67.40,11.46,True
2,2023-2024,3338,66.88,67.30,11.89,True
3,2024-2025,3834,66.85,67.20,11.59,True



 Estado de beca


,Categoría,Estudiantes,Media,Mediana,Desviacion,Grupo suficiente
0,Full,895,67.35,67.60,11.46,True
1,No Scholarship,9354,67.21,67.60,11.82,True
2,Partial,1751,65.30,65.60,11.03,True


### 17. Análisis de relaciones entre variables numéricas
Se presentan correlaciones de Pearson (relación lineal) y Spearman (asociación por rangos) con la calificación final. Se excluyen las categorías de calificación y el indicador de riesgo para evitar relaciones circulares con el resultado.

In [17]:
predictoras_numericas = [c for c in numericas if c != objetivo]
variables_corr = predictoras_numericas + [objetivo]
correlacion_pearson = base[variables_corr].corr(method="pearson")
correlacion_spearman = base[variables_corr].corr(method="spearman")

tabla_correlacion = pd.DataFrame({
    "Variable": [etiquetas[c] for c in predictoras_numericas],
    "Pearson con calificación final": [correlacion_pearson.loc[c,objetivo] for c in predictoras_numericas],
    "Spearman con calificación final": [correlacion_spearman.loc[c,objetivo] for c in predictoras_numericas],
    "Pares válidos": [int(base[[c,objetivo]].dropna().shape[0]) for c in predictoras_numericas]
})
tabla_correlacion["Fuerza absoluta de Spearman"] = tabla_correlacion["Spearman con calificación final"].abs()
tabla_correlacion = tabla_correlacion.sort_values("Fuerza absoluta de Spearman", ascending=False)
display(tabla_correlacion.drop(columns="Fuerza absoluta de Spearman"))

,Variable,Pearson con calificación final,Spearman con calificación final,Pares válidos
8,Calificación anterior (%),0.51,0.54,12000
3,Ingreso familiar anual (INR),0.29,0.37,12000
6,Horas de estudio semanales,0.22,0.24,12000
7,Asistencia (%),0.17,0.18,12000
5,Participación parental,0.09,0.10,12000
1,Tamaño familiar,-0.05,-0.05,12000
2,Número de hermanos,-0.05,-0.05,12000
4,Distancia escolar (km),-0.03,-0.03,12000
0,Edad,0.00,0.00,12000


### 18. Identificación de patrones y hallazgos estadísticos
Se reúnen las relaciones más relevantes observadas y las diferencias de medias entre categorías con al menos 30 estudiantes por grupo. Son hallazgos exploratorios, no pruebas de causalidad.

In [18]:
hallazgos = []
# Relaciones numéricas más fuertes con la calificación final.
for _, fila in tabla_correlacion.head(3).iterrows():
    hallazgos.append({
        "Tipo": "Asociación numérica", "Variable o grupo": fila["Variable"],
        "Resultado": f"Spearman = {fila['Spearman con calificación final']:.3f}",
        "Detalle": f"{int(fila['Pares válidos']):,} pares válidos"
    })

# Diferencias entre grupos categóricos con muestra suficiente.
diferencias_grupos = []
variables_comparacion = ["family_income_bracket", "residence_type", "school_type",
                        "gender", "scholarship_status", "extra_tutoring",
                        "single_parent_household", "has_internet_access",
                        "father_education_level", "mother_education_level"]
for col in variables_comparacion:
    tabla = comparar_grupos(col, minimo=30)
    validos = tabla.loc[tabla["Grupo suficiente"] & tabla["Media"].notna()]
    if len(validos) < 2:
        continue
    maximo = validos.loc[validos["Media"].idxmax()]
    minimo = validos.loc[validos["Media"].idxmin()]
    diferencias_grupos.append({
        "Variable": col, "Descripción": etiquetas[col],
        "Grupo con mayor media": str(maximo["Categoría"]),
        "Media mayor": float(maximo["Media"]), "N mayor": int(maximo["Estudiantes"]),
        "Grupo con menor media": str(minimo["Categoría"]),
        "Media menor": float(minimo["Media"]), "N menor": int(minimo["Estudiantes"]),
        "Diferencia (puntos porcentuales)": float(maximo["Media"] - minimo["Media"])
    })
tabla_diferencias = pd.DataFrame(diferencias_grupos)
if not tabla_diferencias.empty:
    tabla_diferencias = tabla_diferencias.sort_values("Diferencia (puntos porcentuales)", ascending=False)
    print("Diferencias de medias por características (grupos de al menos 30 estudiantes):")
    display(tabla_diferencias)
    for _, r in tabla_diferencias.head(4).iterrows():
        hallazgos.append({
            "Tipo":"Comparación de grupos", "Variable o grupo":r["Descripción"],
            "Resultado":f"Diferencia entre medias: {r['Diferencia (puntos porcentuales)']:.2f} puntos",
            "Detalle": f"{r['Grupo con mayor media']} (n={r['N mayor']}) frente a "
                       f"{r['Grupo con menor media']} (n={r['N menor']})"
        })

print("Hallazgos prioritarios del análisis descriptivo:")
tabla_hallazgos = pd.DataFrame(hallazgos)
display(tabla_hallazgos)
print("Nota: 'mayor diferencia' no implica causa ni significación estadística.")

Diferencias de medias por características (grupos de al menos 30 estudiantes):


,Variable,Descripción,Grupo con mayor media,Media mayor,N mayor,Grupo con menor media,Media menor,N menor,Diferencia (puntos porcentuales)
8,father_education_level,Educación del padre,Doctorate,76.03,361,No Formal Education,60.62,811,15.41
9,mother_education_level,Educación de la madre,Doctorate,75.51,379,No Formal Education,60.40,822,15.12
0,family_income_bracket,Categoría de ingreso familiar,High,76.42,873,Low,62.07,2628,14.35
7,has_internet_access,Acceso a internet,1.0,68.86,5869,0.0,65.04,5951,3.82
5,extra_tutoring,Tutorías adicionales,1,69.48,2762,0,66.19,9238,3.29
4,scholarship_status,Estado de beca,Full,67.35,895,Partial,65.30,1751,2.05
1,residence_type,Zona de residencia,Urban,67.54,5026,Rural,66.37,4224,1.16
6,single_parent_household,Hogar monoparental,0,67.06,10318,1,66.24,1682,0.82
3,gender,Género,Male,66.99,5919,Other,66.37,348,0.62
2,school_type,Tipo de centro educativo,Government-Aided,67.00,2386,Private-Unaided,66.93,4241,0.07


Hallazgos prioritarios del análisis descriptivo:


,Tipo,Variable o grupo,Resultado,Detalle
0,Asociación numérica,Calificación anterior (%),Spearman = 0.535,"12,000 pares válidos"
1,Asociación numérica,Ingreso familiar anual (INR),Spearman = 0.370,"12,000 pares válidos"
2,Asociación numérica,Horas de estudio semanales,Spearman = 0.239,"12,000 pares válidos"
3,Comparación de grupos,Educación del padre,Diferencia entre medias: 15.41 puntos,Doctorate (n=361) frente a No Formal Education...
4,Comparación de grupos,Educación de la madre,Diferencia entre medias: 15.12 puntos,Doctorate (n=379) frente a No Formal Education...
5,Comparación de grupos,Categoría de ingreso familiar,Diferencia entre medias: 14.35 puntos,High (n=873) frente a Low (n=2628)
6,Comparación de grupos,Acceso a internet,Diferencia entre medias: 3.82 puntos,1.0 (n=5869) frente a 0.0 (n=5951)


Nota: 'mayor diferencia' no implica causa ni significación estadística.


### 19. Resumen de resultados del análisis estadístico

En este apartado se reúnen los principales resultados obtenidos durante el análisis exploratorio estadístico del dataset **Student Grade SES**.

Se resumen las características del rendimiento académico, las medidas estadísticas más relevantes, la distribución de las calificaciones y las relaciones observadas entre el desempeño de los estudiantes y sus características socioeconómicas, familiares y académicas.

Los resultados se presentan mediante una tabla consolidada y posteriormente se interpretan para identificar los hallazgos más importantes del análisis.


In [19]:
n = len(base)
resumen_items = [
    {"Aspecto":"Datos analizados",
     "Resultado":f"{n:,} estudiantes; {base.shape[1]} variables; {int(base.isna().sum().sum())} valores nulos",
     "Referencia":"Apartados 2–4"},
    {"Aspecto":"Calificación final",
     "Resultado":f"Media {cal_final.mean():.2f} %, mediana {cal_final.median():.2f} %, "
                 f"desviación {cal_final.std():.2f} puntos",
     "Referencia":"Apartados 5–8 y 12"},
    {"Aspecto":"50 % central del rendimiento",
     "Resultado":f"Entre {cal_final.quantile(.25):.2f} % y {cal_final.quantile(.75):.2f} %",
     "Referencia":"Apartados 8 y 12"},
]

for _, fila in tabla_correlacion.head(2).iterrows():
    resumen_items.append({
        "Aspecto":"Relación estadística destacada",
        "Resultado":f"{fila['Variable']}: Spearman {fila['Spearman con calificación final']:.3f}",
        "Referencia":"Apartados 17–18"
    })

if not tabla_diferencias.empty:
    for _, fila in tabla_diferencias.head(3).iterrows():
        resumen_items.append({
            "Aspecto":"Diferencia entre grupos",
            "Resultado":f"{fila['Descripción']}: {fila['Diferencia (puntos porcentuales)']:.2f} puntos "
                        f"entre {fila['Grupo con mayor media']} y {fila['Grupo con menor media']}",
            "Referencia":"Apartados 13–16 y 18"
        })

for col in ["grade_class", "at_risk_flag"]:
    frec = tablas_frecuencia[col]
    principal = frec.iloc[0]
    resumen_items.append({
        "Aspecto":etiquetas[col],
        "Resultado":f"Categoría más frecuente: {principal['Categoría']} "
                    f"({principal['Porcentaje (%)']:.1f} %; n={int(principal['Frecuencia']):,})",
        "Referencia":"Apartados 10–12"
    })

resumen_items.append({
    "Aspecto":"Limitaciones",
    "Resultado":"Datos previamente imputados; resultados descriptivos; "
                "clase de nota y riesgo pueden derivarse de la nota final",
    "Referencia":"ETL y apartados 4, 12 y 17"
})

resumen_eda = pd.DataFrame(resumen_items)
display(resumen_eda)

# Redacción breve del resumen, preparada para revisar después de ver las tablas.
parrafos_resumen = [
    f"Se analizaron {n:,} registros y {base.shape[1]} variables del dataset limpio. "
    f"La calificación final presentó una media de {cal_final.mean():.2f} %, "
    f"una mediana de {cal_final.median():.2f} % y "
    f"una desviación estándar de {cal_final.std():.2f} puntos.",
]
if not tabla_correlacion.empty:
    f = tabla_correlacion.iloc[0]
    parrafos_resumen.append(
        f"La asociación numérica de mayor magnitud con la nota final, dentro de las variables "
        f"examinadas, fue {f['Variable']} (Spearman = {f['Spearman con calificación final']:.3f})."
    )
if not tabla_diferencias.empty:
    f = tabla_diferencias.iloc[0]
    parrafos_resumen.append(
        f"Entre las comparaciones seleccionadas, {f['Descripción']} presentó una separación "
        f"de {f['Diferencia (puntos porcentuales)']:.2f} puntos entre las medias de los grupos "
        f"{f['Grupo con mayor media']} y {f['Grupo con menor media']}, "
        f"considerando únicamente grupos con al menos 30 registros."
    )
parrafos_resumen.append(
    "Estos hallazgos describen asociaciones, no causas; deben considerarse los valores imputados "
    "y la posible dependencia de las variables de clasificación académica respecto de la nota final."
)
display(Markdown("**Resumen narrativo preliminar (basado en el CSV ejecutado):**\n\n" + "\n\n".join(parrafos_resumen)))

,Aspecto,Resultado,Referencia
0,Datos analizados,"12,000 estudiantes; 29 variables; 0 valores nulos",Apartados 2–4
1,Calificación final,"Media 66.94 %, mediana 67.30 %, desviación 11....",Apartados 5–8 y 12
2,50 % central del rendimiento,Entre 60.10 % y 74.60 %,Apartados 8 y 12
3,Relación estadística destacada,Calificación anterior (%): Spearman 0.535,Apartados 17–18
4,Relación estadística destacada,Ingreso familiar anual (INR): Spearman 0.370,Apartados 17–18
5,Diferencia entre grupos,Educación del padre: 15.41 puntos entre Doctor...,Apartados 13–16 y 18
6,Diferencia entre grupos,Educación de la madre: 15.12 puntos entre Doct...,Apartados 13–16 y 18
7,Diferencia entre grupos,Categoría de ingreso familiar: 14.35 puntos en...,Apartados 13–16 y 18
8,Categoría de calificación,"Categoría más frecuente: C (51.2 %; n=6,147)",Apartados 10–12
9,Indicador de riesgo académico,"Categoría más frecuente: 0 (98.2 %; n=11,782)",Apartados 10–12


**Resumen narrativo preliminar (basado en el CSV ejecutado):**

Se analizaron 12,000 registros y 29 variables del dataset limpio. La calificación final presentó una media de 66.94 %, una mediana de 67.30 % y una desviación estándar de 11.70 puntos.

La asociación numérica de mayor magnitud con la nota final, dentro de las variables examinadas, fue Calificación anterior (%) (Spearman = 0.535).

Entre las comparaciones seleccionadas, Educación del padre presentó una separación de 15.41 puntos entre las medias de los grupos Doctorate y No Formal Education, considerando únicamente grupos con al menos 30 registros.

Estos hallazgos describen asociaciones, no causas; deben considerarse los valores imputados y la posible dependencia de las variables de clasificación académica respecto de la nota final.


**Interpretación de resultados:**

El análisis estadístico se realizó sobre **12.000 estudiantes y 29 variables**, utilizando el dataset previamente limpiado y sin valores nulos.

La calificación final presentó una media de **66,94 %** y una mediana de **67,30 %**, lo que indica valores centrales similares. Además, el 50 % de las calificaciones se concentró entre **60,10 % y 74,60 %**.

Entre las variables numéricas analizadas, la calificación del año anterior presentó la asociación positiva más destacada con el rendimiento final (Spearman = **0,535**), seguida del ingreso familiar anual (Spearman = **0,370**).

Las comparaciones entre grupos mostraron diferencias en el rendimiento promedio según características familiares y socioeconómicas. Se observaron separaciones de hasta **15,41 puntos** según la educación del padre, **15,12 puntos** según la educación de la madre y **14,35 puntos** según la categoría de ingreso familiar.

La categoría de calificación más frecuente fue **C**, con **6.147 estudiantes (51,2 %)**. Por su parte, el indicador de riesgo académico presentó una concentración del **98,2 %** en la categoría 0.

En conjunto, los resultados muestran asociaciones entre el rendimiento académico y determinadas características de los estudiantes. Sin embargo, estas relaciones son descriptivas y no permiten establecer causalidad.



### 20. Conclusiones de la Parte 02: Análisis estadístico

El análisis exploratorio estadístico permitió conocer las principales características del rendimiento académico de los estudiantes e identificar relaciones con factores familiares, socioeconómicos y académicos.

A partir de los resultados obtenidos, se establecen las siguientes conclusiones:

1. **Rendimiento académico general:** la media y la mediana de las calificaciones finales fueron similares, situándose alrededor del 67 %. Esto permite identificar el comportamiento central del rendimiento académico de los estudiantes.

2. **Importancia del rendimiento anterior:** la calificación del año anterior presentó una asociación positiva con la calificación final (Spearman = 0,535), lo que indica que los estudiantes con mejores resultados anteriores tienden a presentar un mayor rendimiento actual.

3. **Relación con las condiciones socioeconómicas:** el ingreso familiar anual mostró una asociación positiva con el rendimiento académico (Spearman = 0,370). Asimismo, se observaron diferencias entre categorías de ingresos, lo que sugiere que las condiciones económicas pueden estar relacionadas con el desempeño estudiantil.

4. **Diferencias según características familiares:** el nivel educativo de los padres presentó diferencias en las calificaciones promedio entre grupos. Estos resultados permiten identificar posibles relaciones entre el entorno familiar y el rendimiento académico, sin afirmar que exista una relación causal.

5. **Distribución de las calificaciones:** la categoría C concentró el 51,2 % de los estudiantes. Además, el indicador de riesgo académico mostró una distribución muy desigual entre sus categorías, aspecto que deberá considerarse en análisis posteriores.

6. **Limitaciones del análisis:** algunas variables contienen valores previamente imputados durante el ETL. Además, las categorías de calificación y riesgo académico podrían derivarse de la calificación final, por lo que no deben tratarse automáticamente como variables explicativas independientes.

### Conclusión general

El análisis estadístico permitió identificar patrones y asociaciones relevantes entre el rendimiento académico y las características de los estudiantes.

Entre los hallazgos más importantes destacan la relación positiva con la calificación del año anterior, la asociación con los ingresos familiares y las diferencias de rendimiento observadas entre determinados grupos socioeconómicos y familiares.

Estos resultados constituyen una base para continuar con la **Parte 03: Análisis exploratorio visual (EDA)**, donde se examinarán gráficamente las distribuciones, relaciones y patrones identificados.

Las asociaciones encontradas deberán interpretarse con precaución, considerando las limitaciones de los datos y evitando establecer conclusiones causales únicamente a partir de estadísticas descriptivas.
